In [7]:
from kafka import KafkaConsumer
import json
import matplotlib.pyplot as plt
import time
import os

In [8]:
# Initialize data storage for plotting
timestamps = []
predictions = []
conf_no_aeration = []
conf_aeration = []
SAVE_PATH = "/opt/local/water_prediction_plot.png"
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

In [9]:
# Kafka configuration
def initialize_consumer():
    kafka_topic = "prediction_topic"
    kafka_bootstrap_servers = ["localhost:9092"]

    # Create Kafka consumer
    consumer = KafkaConsumer(
        kafka_topic,
        bootstrap_servers=kafka_bootstrap_servers,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='latest',
        enable_auto_commit=True
    )
    return consumer

In [10]:
# Receive all published messages and update plot
def update_plot(consumer):
    try:
        for message in consumer:
            # Parse the message
            sensor_data = message.value
            print(f"Received: {sensor_data}")

            # Update data storage
            timestamps.append(sensor_data['timestamp'])
            predictions.append(sensor_data['prediction'])
            conf_no_aeration.append(sensor_data['proba_no_areation'])
            conf_aeration.append(sensor_data['proba_areation'])

            # Keep only the last 100 entries for plotting
            if len(timestamps) > 100:
                timestamps.pop(0)
                predictions.pop(0)
                conf_no_aeration.pop(0)
                conf_aeration.pop(0)

            fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 12), sharex=True)

            # --- 3.3.1: Estado de aireación ---
            ax1.step(timestamps, predictions, where='post', color='blue', linewidth=2)
            ax1.set_title("Estado de Aireación por Timestamp (1=Sí, 0=No)")
            ax1.set_ylabel("Estado")
            ax1.grid(True, alpha=0.3)

            # --- 3.3.2: Confianza Clase Aireación (cuando es escogida) ---
            ax2.step(timestamps, conf_aeration, color='green', label='Confianza Aireación')
            ax2.set_title("Confianza: Clase Aireación (Solo cuando es la clase predicha)")
            ax2.set_ylabel("Probabilidad")
            ax2.set_ylim(0, 1.0) # La confianza siempre será > 0.5 si fue la elegida
            ax2.grid(True, alpha=0.3)

            # --- 3.3.3: Confianza Clase No Aireación (cuando es escogida) ---
            ax3.step(timestamps, conf_no_aeration, color='red', label='Confianza No Aireación')
            ax3.set_title("Confianza: Clase No Aireación (Solo cuando es la clase predicha)")
            ax3.set_ylabel("Probabilidad")
            ax3.set_xlabel("Timestamp")
            ax3.set_ylim(0, 1.0)
            ax3.grid(True, alpha=0.3)

            ax4.remove()

            # Formatear el eje X para que no se amontonen los timestamps
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()

            # Save the plot as an image
            plt.savefig(SAVE_PATH)
            plt.close()

            break  # Process one message at a time
    except KeyboardInterrupt:
        print("Stopped consuming messages.")
        consumer.close()

In [11]:
# Listas globales para cada proceso (se inicializan vacías al arrancar el worker)
timestamps = []
predictions = []
conf_no_aeration = []
conf_aeration = []

def run_consumer(instance_id):
    save_path = f"plot_worker_{instance_id}.png"
    
    # Inicialización del consumidor real aquí
    # consumer = initialize_consumer() 
    
    print(f"[Worker-{instance_id}] Subscribed and saving to {save_path}")
    
    try:
        while True:
            # En entorno real: for message in consumer:
            # Simulamos la recepción de un mensaje para el ejemplo:
            message_value = {
                'timestamp': time.strftime("%H:%M:%S"),
                'prediction': 1,
                'proba_no_areation': 0.1,
                'proba_aeration': 0.9
            }
            
            # 1. Print de lo que escucha
            print(f"[Worker-{instance_id}] Received: {message_value}")

            # 2. Actualización de datos
            timestamps.append(message_value['timestamp'])
            predictions.append(message_value['prediction'])
            conf_no_aeration.append(message_value['proba_no_areation'])
            conf_aeration.append(message_value['proba_aeration'])

            if len(timestamps) > 100:
                timestamps.pop(0)
                predictions.pop(0)
                conf_no_aeration.pop(0)
                conf_aeration.pop(0)

            # 3. Generación y guardado del Plot
            fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 12))

            ax1.step(timestamps, predictions, where='post', color='blue', linewidth=2)
            ax1.set_title(f"Aireación (Worker {instance_id})")
            ax1.grid(True, alpha=0.3)

            ax2.step(timestamps, conf_aeration, color='green')
            ax2.set_title("Confianza Aireación")
            ax2.set_ylim(0, 1.0)
            ax2.grid(True, alpha=0.3)

            ax3.step(timestamps, conf_no_aeration, color='red')
            ax3.set_title("Confianza No Aireación")
            ax3.set_ylim(0, 1.0)
            ax3.grid(True, alpha=0.3)

            ax4.remove()
            
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()

            # Guardar el plot
            plt.savefig(save_path)
            plt.close(fig)

            time.sleep(2) # Pausa para no saturar el disco de imágenes

    except KeyboardInterrupt:
        print(f"Worker-{instance_id} stopped.")

# 👂 Escuchamos

In [12]:
# consumer = initialize_consumer()
# print("Subscribed to Kafka topic 'water_quality'.")

# try:
#     while True:
#         update_plot(consumer)
        
# except KeyboardInterrupt:
#     print("Stopped visualization.")
#     consumer.close()

In [ ]:
import multiprocessing
import time
import matplotlib.pyplot as plt
num_consumers = 2
processes = []

for i in range(num_consumers):
    # Cada proceso recibe su ID para diferenciar el archivo de salida
    p = multiprocessing.Process(target=run_consumer, args=(i+1,))
    p.start()
    processes.append(p)

try:
    for p in processes:
        p.join()
except KeyboardInterrupt:
    print("\nStopping all consumers...")
    for p in processes:
        p.terminate()

[Worker-1] Subscribed and saving to plot_worker_1.png
[Worker-1] Received: {'timestamp': '17:20:08', 'prediction': 1, 'proba_no_areation': 0.1, 'proba_aeration': 0.9}[Worker-2] Subscribed and saving to plot_worker_2.png

[Worker-2] Received: {'timestamp': '17:20:08', 'prediction': 1, 'proba_no_areation': 0.1, 'proba_aeration': 0.9}


[Worker-1] Received: {'timestamp': '17:20:10', 'prediction': 1, 'proba_no_areation': 0.1, 'proba_aeration': 0.9}
[Worker-2] Received: {'timestamp': '17:20:10', 'prediction': 1, 'proba_no_areation': 0.1, 'proba_aeration': 0.9}
[Worker-1] Received: {'timestamp': '17:20:13', 'prediction': 1, 'proba_no_areation': 0.1, 'proba_aeration': 0.9}
[Worker-2] Received: {'timestamp': '17:20:13', 'prediction': 1, 'proba_no_areation': 0.1, 'proba_aeration': 0.9}
[Worker-1] Received: {'timestamp': '17:20:15', 'prediction': 1, 'proba_no_areation': 0.1, 'proba_aeration': 0.9}
[Worker-2] Received: {'timestamp': '17:20:15', 'prediction': 1, 'proba_no_areation': 0.1, 'proba_aeration': 0.9}
[Worker-2] Received: {'timestamp': '17:20:17', 'prediction': 1, 'proba_no_areation': 0.1, 'proba_aeration': 0.9}
[Worker-1] Received: {'timestamp': '17:20:17', 'prediction': 1, 'proba_no_areation': 0.1, 'proba_aeration': 0.9}
[Worker-2] Received: {'timestamp': '17:20:20', 'prediction': 1, 'proba_no_areation': 0.1, 'prob